# Random Forest Genre Classifier for FMA Small or Medium

This notebook builds a random forest genre classifier using the precomputed tabular audio features in `fma_metadata/features.csv` joined onto the cleaned per-split CSVs produced by [preprocess_spectrograms.ipynb](preprocess_spectrograms.ipynb).

Set `SUBSET` below to `"small"` (≈8 000 tracks, 8 genres) or `"medium"` (≈25 000 tracks, 16 genres). Both cleaned CSVs are produced by the preprocessing notebook — only the source data and the genre count differ.

Pipeline:

1. Resolve the project paths and load the chosen subset's per-split CSV from `fma_preprocessed/`.
2. Join each split onto the handcrafted FMA features to build the training, validation, and test matrices.
3. Optionally cap each split per genre to keep experiments manageable and balanced.
4. Train a random forest over one or more `n_estimators` settings.
5. Save the checkpoint with the best validation accuracy.
6. Evaluate the best model on the held-out test split.

If `fma_preprocessed/` is missing, run [preprocess_spectrograms.ipynb](preprocess_spectrograms.ipynb) first.

## 1. Setup

If the imports below fail in your Jupyter kernel, run this once in a notebook cell, restart the kernel, then continue:

```python
%pip install pandas numpy scikit-learn
```


In [ ]:
from __future__ import annotations

import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Pick which cleaned FMA subset to train on. preprocess_spectrograms.ipynb writes both.
SUBSET = "small"  # or "medium"
if SUBSET not in {"small", "medium"}:
    raise ValueError("SUBSET must be either 'small' or 'medium'.")

# Resolve the dataset paths relative to either the current working directory or its parent.
PROJECT_CANDIDATES = [Path.cwd(), Path.cwd().parent]

for candidate in PROJECT_CANDIDATES:
    features_path = candidate / "fma_metadata" / "features.csv"
    preprocessed_dir = candidate / "fma_preprocessed"
    if features_path.exists() and preprocessed_dir.exists():
        PROJECT_DIR = candidate.resolve()
        FEATURES_PATH = features_path.resolve()
        PREPROCESSED_DIR = preprocessed_dir.resolve()
        break
else:
    raise FileNotFoundError(
        "Could not find fma_metadata/features.csv and fma_preprocessed/ from the current "
        "directory or its parent. Run preprocess_spectrograms.ipynb first to produce the "
        "cleaned per-split CSVs in fma_preprocessed/."
    )

print({
    "SUBSET": SUBSET,
    "PROJECT_DIR": str(PROJECT_DIR),
    "FEATURES_PATH": str(FEATURES_PATH),
    "PREPROCESSED_DIR": str(PREPROCESSED_DIR),
})

## 2. Load Preprocessed Splits + Features

The cleaning is already done by [preprocess_spectrograms.ipynb](preprocess_spectrograms.ipynb): it filtered to `fma_small`, dropped tracks with no genre, dropped undecodable MP3s, and removed the `FAILED` track ids from `creation.ipynb`. Here we just load each cleaned split and join on the handcrafted features the random forest consumes.

In [ ]:
def flatten_columns(columns: pd.Index) -> list[str]:
    return ["__".join(str(part) for part in column) for column in columns]


# features.csv has a three-row header; flatten it once so every column is a single string.
features = pd.read_csv(FEATURES_PATH, index_col=0, header=[0, 1, 2])
features.columns = flatten_columns(features.columns)
features.index = features.index.astype(int)


def load_split(split: str) -> pd.DataFrame:
    """Read the cleaned per-split CSV for the chosen SUBSET and join on the handcrafted features."""
    split_path = PREPROCESSED_DIR / f"tracks_clean_{SUBSET}_{split}.csv"
    if not split_path.exists():
        raise FileNotFoundError(
            f"Missing {split_path}. Run preprocess_spectrograms.ipynb to produce the "
            f"{SUBSET} CSVs."
        )
    frame = pd.read_csv(split_path)
    frame["track_id"] = frame["track_id"].astype(int)
    frame = frame.set_index("track_id")
    return frame.join(features, how="inner")


train_raw = load_split("training")
val_raw = load_split("validation")
test_raw = load_split("test")

# Concatenate for a quick post-cleaning summary.
metadata = pd.concat([train_raw, val_raw, test_raw])
metadata.head()

In [ ]:
# Non-feature columns inherited from tracks_clean.csv. Everything else is a feature.
METADATA_COLUMNS = {
    "subset", "split", "genre", "label",
    "duration", "bit_rate", "title", "artist", "audio_path",
}

print(f"Usable rows after preprocessing: {len(metadata):,}")
print(metadata["split"].value_counts())
print(metadata["genre"].value_counts().sort_index())
print(f"Feature columns: {sum(1 for c in metadata.columns if c not in METADATA_COLUMNS)}")

## 3. Pick a Manageable Balanced Split

The splits are already the official FMA ones. The caps below let you shrink each split per genre when you want a quicker experiment; leave them as `None` to use everything.

In [ ]:
# These caps keep classes balanced while letting you shrink the experiment when needed.
MAX_TRAIN_PER_GENRE = None
MAX_EVAL_PER_GENRE = None


def cap_per_genre(frame: pd.DataFrame, max_per_genre: int | None) -> pd.DataFrame:
    if frame.empty:
        return frame.reset_index(drop=False)
    if max_per_genre is None:
        return frame.sample(frac=1, random_state=SEED).reset_index(drop=False)

    sampled_groups = [
        group.sample(min(len(group), max_per_genre), random_state=SEED)
        for _, group in frame.groupby("genre", sort=False)
    ]
    return (
        pd.concat(sampled_groups)
        .sample(frac=1, random_state=SEED)
        .reset_index(drop=False)
    )


train_df = cap_per_genre(train_raw, MAX_TRAIN_PER_GENRE)
val_df = cap_per_genre(val_raw, MAX_EVAL_PER_GENRE)
test_df = cap_per_genre(test_raw, MAX_EVAL_PER_GENRE)

# Prefer the genre_to_idx mapping written by preprocess_spectrograms.ipynb so this notebook
# uses the same label ids as the CNN/transformer notebooks. Falls back to deriving from train.
genre_map_path = PREPROCESSED_DIR / f"genre_to_idx_{SUBSET}.csv"
if genre_map_path.exists():
    genre_map_df = pd.read_csv(genre_map_path)
    genre_to_idx = dict(zip(genre_map_df["genre"], genre_map_df["label"].astype(int)))
else:
    genres_sorted = sorted(train_df["genre"].unique())
    genre_to_idx = {g: i for i, g in enumerate(genres_sorted)}
idx_to_genre = {idx: genre for genre, idx in genre_to_idx.items()}
genres = [idx_to_genre[i] for i in sorted(idx_to_genre)]

for frame in (train_df, val_df, test_df):
    frame["label"] = frame["genre"].map(genre_to_idx).astype(int)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(genre_to_idx)

## 4. Build Tabular Matrices


In [ ]:
def split_features_and_labels(frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, list[str]]:
    feature_columns = [
        column
        for column in frame.columns
        if column not in METADATA_COLUMNS and column != "track_id"
    ]
    x = frame[feature_columns].to_numpy(dtype=np.float32)
    y = frame["label"].to_numpy(dtype=np.int64)
    return x, y, feature_columns


X_train, y_train, feature_columns = split_features_and_labels(train_df)
X_val, y_val, _ = split_features_and_labels(val_df)
X_test, y_test, _ = split_features_and_labels(test_df)

X_train.shape, X_val.shape, X_test.shape

## 5. Train the Random Forest

The cell below tries multiple forest sizes and keeps the model with the best validation accuracy.

`ccp_alpha` is the direct post-pruning control. `max_depth`, `min_samples_split`, `min_samples_leaf`, and `max_leaf_nodes` also regularize tree growth and are usually the first knobs to tune.


In [ ]:
# Tune these settings if you want to explore different random forest configurations.
N_ESTIMATOR_OPTIONS = [250]
MAX_DEPTH = 10
MAX_FEATURES = None
MIN_SAMPLES_SPLIT = 5
MIN_SAMPLES_LEAF = 1
MAX_LEAF_NODES = None
CCP_ALPHA = 0.0

best_val_acc = -1.0
best_model = None
best_n_estimators = None
history = []
best_model_path = PROJECT_DIR / "code" / f"best_random_forest_{SUBSET}.pkl"

for n_estimators in N_ESTIMATOR_OPTIONS:
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=MAX_DEPTH,
        max_features=MAX_FEATURES,
        min_samples_split=MIN_SAMPLES_SPLIT,
        min_samples_leaf=MIN_SAMPLES_LEAF,
        max_leaf_nodes=MAX_LEAF_NODES,
        ccp_alpha=CCP_ALPHA,
        n_jobs=-1,
        random_state=SEED,
    )
    model.fit(X_train, y_train)

    train_predictions = model.predict(X_train)
    val_predictions = model.predict(X_val)
    train_acc = accuracy_score(y_train, train_predictions)
    val_acc = accuracy_score(y_val, val_predictions)

    history.append({
        "n_estimators": n_estimators,
        "train_acc": train_acc,
        "val_acc": val_acc,
    })

    print(
        f"n_estimators {n_estimators:4d} | "
        f"train acc {train_acc:.3f} | "
        f"val acc {val_acc:.3f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model = model
        best_n_estimators = n_estimators
        with best_model_path.open("wb") as handle:
            pickle.dump({
                "model": best_model,
                "genre_to_idx": genre_to_idx,
                "idx_to_genre": idx_to_genre,
                "feature_columns": feature_columns,
                "config": {
                    "subset": SUBSET,
                    "preprocessed_dir": str(PREPROCESSED_DIR),
                    "max_train_per_genre": MAX_TRAIN_PER_GENRE,
                    "max_eval_per_genre": MAX_EVAL_PER_GENRE,
                    "n_estimators": best_n_estimators,
                    "max_depth": MAX_DEPTH,
                    "max_features": MAX_FEATURES,
                    "min_samples_split": MIN_SAMPLES_SPLIT,
                    "min_samples_leaf": MIN_SAMPLES_LEAF,
                    "max_leaf_nodes": MAX_LEAF_NODES,
                    "ccp_alpha": CCP_ALPHA,
                    "seed": SEED,
                },
                "history": history,
            }, handle)

pd.DataFrame(history)

## 5b. Hyperparameter Tuning

Sweep a grid of values for each random forest knob and pick the configuration with the best validation accuracy. The grids below are small on purpose so the sweep stays tractable; widen them when you have more time. The best model from the sweep replaces the checkpoint saved above only if it beats the current best validation accuracy.

In [ ]:
from itertools import product

# Edit these grids to control the search space. Each list is the set of values to try for that hyperparameter.
PARAM_GRID = {
    "n_estimators": [100, 250, 500],
    "max_depth": [10, 20, None],
    "max_features": ["sqrt", "log2", None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_leaf_nodes": [None, 256],
    "ccp_alpha": [0.0, 1e-4],
}

param_names = list(PARAM_GRID.keys())
combinations = list(product(*(PARAM_GRID[name] for name in param_names)))
print(f"Trying {len(combinations)} configurations...")

tuning_history = []
tuned_best_val_acc = -1.0
tuned_best_model = None
tuned_best_params = None

for idx, values in enumerate(combinations, start=1):
    params = dict(zip(param_names, values))
    model = RandomForestClassifier(
        **params,
        n_jobs=-1,
        random_state=SEED,
    )
    model.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, model.predict(X_train))
    val_acc = accuracy_score(y_val, model.predict(X_val))

    tuning_history.append({**params, "train_acc": train_acc, "val_acc": val_acc})

    print(
        f"[{idx:3d}/{len(combinations)}] "
        + " ".join(f"{name}={params[name]}" for name in param_names)
        + f" | train {train_acc:.3f} | val {val_acc:.3f}"
    )

    if val_acc > tuned_best_val_acc:
        tuned_best_val_acc = val_acc
        tuned_best_model = model
        tuned_best_params = params

tuning_results = (
    pd.DataFrame(tuning_history)
    .sort_values("val_acc", ascending=False)
    .reset_index(drop=True)
)

print("\nBest configuration from sweep:")
for name, value in tuned_best_params.items():
    print(f"  {name}: {value}")
print(f"  validation accuracy: {tuned_best_val_acc:.3f}")

# Promote the tuned model to the saved checkpoint only if it beats the previous best.
if tuned_best_val_acc > best_val_acc:
    print(f"\nTuned model improved val acc {best_val_acc:.3f} -> {tuned_best_val_acc:.3f}. Updating checkpoint.")
    best_val_acc = tuned_best_val_acc
    best_model = tuned_best_model
    best_n_estimators = tuned_best_params["n_estimators"]
    with best_model_path.open("wb") as handle:
        pickle.dump({
            "model": best_model,
            "genre_to_idx": genre_to_idx,
            "idx_to_genre": idx_to_genre,
            "feature_columns": feature_columns,
            "config": {
                "subset": SUBSET,
                "preprocessed_dir": str(PREPROCESSED_DIR),
                "max_train_per_genre": MAX_TRAIN_PER_GENRE,
                "max_eval_per_genre": MAX_EVAL_PER_GENRE,
                **tuned_best_params,
                "seed": SEED,
            },
            "history": history,
            "tuning_history": tuning_history,
        }, handle)
else:
    print(f"\nNo tuned model beat the existing best ({best_val_acc:.3f}); checkpoint unchanged.")

tuning_results.head(10)

## 6. Test Evaluation


In [ ]:
# Reload the best validation checkpoint before measuring final test performance.
with best_model_path.open("rb") as handle:
    checkpoint = pickle.load(handle)

best_model = checkpoint["model"]
test_predictions = best_model.predict(X_test)
test_acc = accuracy_score(y_test, test_predictions)

print(f"Best validation accuracy: {best_val_acc:.3f} at n_estimators={best_n_estimators}")
print(f"Test accuracy: {test_acc:.3f}")


## 7. Inspect Predictions


In [ ]:
pd.DataFrame({
    "true": [idx_to_genre[int(label)] for label in y_test[:16]],
    "predicted": [idx_to_genre[int(prediction)] for prediction in test_predictions[:16]],
})


In [ ]:
print("Classification report:")
print(classification_report(y_test, test_predictions, target_names=genres, digits=3))

confusion = confusion_matrix(y_test, test_predictions)
confusion_df = pd.DataFrame(confusion, index=genres, columns=genres)
confusion_df


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 10))
image = ax.imshow(confusion, cmap="Blues")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

ax.set_title("Confusion Matrix")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_xticks(range(len(genres)))
ax.set_yticks(range(len(genres)))
ax.set_xticklabels(genres, rotation=90)
ax.set_yticklabels(genres)

for i in range(confusion.shape[0]):
    for j in range(confusion.shape[1]):
        ax.text(j, i, confusion[i, j], ha="center", va="center", color="black", fontsize=8)

plt.tight_layout()
plt.show()
